# Chapter 4: Discrete Behavior Cloning

This Colab follows the Chapter 4 manuscript end to end: it reproduces the
regression trap of section 4.2, fits the action tokenizer of section 4.3,
trains **all three** action heads of section 4.4 on the SO-101
demonstrations, and produces every figure the chapter attributes to code.

Runtime: **Runtime > Change runtime type > GPU**. On CPU the
autoregressive head's serial decode makes section 4.6 impractically slow.

In [ ]:
# Colab setup: install the three public chapter packages from GitHub.
import subprocess
import sys

# Switch CHAPTER_4_REF to 'main' once the Chapter 4 branch is merged.
CHAPTER_4_REF = 'ch04-production-review'

if 'google.colab' in sys.modules:
    organization = 'Large-Robotics-Models-From-Scratch'
    requirements = [
        f'lrm-ch02[data] @ git+https://github.com/{organization}/lrm-code-chapter-2.git@main',
        f'lrm-ch03 @ git+https://github.com/{organization}/lrm-code-chapter-3.git@main',
        f'lrm-ch04[data] @ git+https://github.com/{organization}/lrm-code-chapter-4.git@{CHAPTER_4_REF}',
    ]
    result = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '--quiet', *requirements],
        text=True, capture_output=True,
    )
    if result.returncode:
        detail = '\n'.join(
            part for part in (result.stdout, result.stderr) if part
        )
        raise RuntimeError(
            f'Chapter package installation failed:\n{detail}'
        )
    print('Installed Chapter 2, Chapter 3, and Chapter 4 packages.')

In [ ]:
import math
import os

import numpy as np
import torch
import matplotlib.pyplot as plt

from ch04.constants import ACTION_BINS, ACTION_DIM, ACTION_HORIZON

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
CHECKPOINT_ROOT = ('/content/ch04-checkpoints'
                   if os.path.isdir('/content') else './ch04-checkpoints')
print('device:', device)
print('checkpoints:', CHECKPOINT_ROOT)
print(f'label grid: H={ACTION_HORIZON}, D={ACTION_DIM}, B={ACTION_BINS}')
print(f'uniform cross-entropy at init: ln({ACTION_BINS}) = '
      f'{math.log(ACTION_BINS):.3f}')

## 4.2 The multimodal regression trap

The observation carries no clue about which of the two demonstrated modes
the expert took, so mean squared error has nothing to do but average them.
A two-component mixture, fit on the same data by maximum likelihood, keeps
both. This cell is figure 4.4.

In [ ]:
from ch04.diagnostics import plot_bimodal_comparison
from ch04.exercises import (make_bimodal_actions, train_gmm_baseline,
                            train_mse_baseline)

observations, expert_actions = make_bimodal_actions()
mse_model, mse_history = train_mse_baseline()
mixture, gmm_history = train_gmm_baseline()
with torch.no_grad():
    collapsed = mse_model(torch.zeros(1, 1)).item()
    log_weights, means, sigmas = mixture(torch.zeros(1, 1))

print(f'MSE prediction:  {collapsed:+.3f}  (expert modes are -1 and +1)')
print('mixture means:  ', [round(v, 3) for v in means[0].tolist()])
print('mixture weights:', [round(v, 3) for v in log_weights.exp()[0].tolist()])

# Figure 4.4: the regressor lands in the valley, the mixture does not.
plot_bimodal_comparison(expert_actions.numpy(), collapsed, mixture=mixture)
plt.show()

## 4.3 Fit the action tokenizer

Normalization statistics and tokenizer bounds come from the **training**
episodes only. The split is episode-disjoint, so no held-out frame
influences the q01/q99 range the bins are laid out over.

In [ ]:
from ch04 import ActionTokenizer
from ch04.data import (DEFAULT_DATASET_ID, collect_normalized_actions,
                       make_chunked_dataloaders)

train_loader, validation_loader, stats = make_chunked_dataloaders(
    DEFAULT_DATASET_ID, batch_size=4, validation_fraction=0.1)
# None uses every training frame through the fast Arrow action column.
# Set an integer only for a bounded loader-based smoke fit.
TOKENIZER_FIT_BATCHES = None
normalized_actions = collect_normalized_actions(
    train_loader, stats, max_batches=TOKENIZER_FIT_BATCHES)
tokenizer = ActionTokenizer.fit(normalized_actions)

example = normalized_actions[0]
bins = tokenizer.encode(example)
decoded = tokenizer.decode(bins)
print('normalized action:', np.round(example, 4))
print('bin ids:          ', bins, '(a separate 256-entry AR table)')
print('midpoint decode:  ', np.round(decoded, 4))
print('max round-trip error:', np.abs(decoded - example).max(),
      '<= half a bin width')
print('train episodes:     ', len(train_loader.dataset.episodes))
print('validation episodes:', validation_loader.dataset.episodes)

In [ ]:
# One target is an H x D grid: H future timesteps, D control dimensions.
demo_grid = torch.arange(ACTION_HORIZON * ACTION_DIM).reshape(
    1, ACTION_HORIZON, ACTION_DIM)
print('target grid:', tuple(demo_grid.shape))
print('parallel action positions:', ACTION_HORIZON,
      '(one per timestep, SmolVLA-style)')
print('autoregressive scalar tokens:', ACTION_HORIZON * ACTION_DIM)
print('first two timestep vectors:', demo_grid[0, :2].tolist())

## 4.4 Build the three action heads

`build_action_head` is the same factory the `ch04-train` command uses.
Each head gets its **own** freshly initialized Chapter 3 backbone: sharing
one would let the second head start from a trunk the first had already
adapted, and the comparison in section 4.6 would mean nothing.

In [ ]:
from ch03 import VLABackbone
from ch04.cli import build_action_head

HEAD_NAMES = ('factorized', 'autoregressive', 'parallel')


def make_policy(name):
    """Return a fresh (backbone, head) pair for one head design."""
    backbone = VLABackbone().to(device)
    head = build_action_head(name, backbone).to(device)
    return backbone, head


print('heads:', ', '.join(HEAD_NAMES))
print('each gets its own FactorizedActionHead / '
      'AutoregressiveActionHead / ParallelDecodeActionHead instance')

## 4.5 One shared training and evaluation routine

Every head optimizes the same categorical target: labels shaped `[H, D]`,
logits shaped `[H, D, B]`, masked label-smoothed cross-entropy. Only the
call signature differs, and `action_head_logits` absorbs that, so one
function trains and evaluates all three.

The returned dictionary keeps the batch and prepared inputs so the
section 4.6 figures can re-use exactly the observation each head was
scored on.

In [ ]:
import itertools

from ch04.data import action_targets, prepare_batch
from ch04.decoding import (decode_action_chunk, evaluate_open_loop,
                           evaluation_mode, sample_action_grids)
from ch04.diagnostics import (plot_action_distribution,
                              plot_chunk_comparison, plot_joint_support,
                              temporal_jitter, within_expert_support)
from ch04.losses import masked_token_cross_entropy
from ch04.train import action_head_logits, train_action_head


def run_head_experiment(name, steps=10, samples=64, eval_batches=1):
    backbone, head = make_policy(name)
    history = train_action_head(
        head, backbone, train_loader, stats, tokenizer, device,
        total_steps=steps, warmup_steps=min(5, steps - 1),
        log_every=max(1, steps // 20), checkpoint_every=steps,
        validation_loader=validation_loader,
        checkpoint_dir=f'{CHECKPOINT_ROOT}/{name}')

    batch = next(iter(validation_loader))
    model_inputs = prepare_batch(batch, stats, device, backbone)
    target_bins, token_pad = action_targets(
        batch, stats, tokenizer, device)
    with torch.no_grad(), evaluation_mode(backbone), evaluation_mode(head):
        logits = action_head_logits(
            head, backbone, model_inputs, target_bins)
        validation_ce = masked_token_cross_entropy(
            logits, target_bins, token_pad).item()

    # Each head samples through its own inference path, so the AR draws
    # carry the conditioning the two parallel heads lack.
    sampled_grids = sample_action_grids(
        head, backbone, model_inputs, n_samples=samples)
    prediction = decode_action_chunk(
        head, backbone, model_inputs, tokenizer, stats,
        strategy='argmax').cpu()
    metrics = evaluate_open_loop(
        head, itertools.islice(validation_loader, eval_batches),
        tokenizer, stats, backbone, device)

    expert = torch.as_tensor(batch['action']).float()
    expert_pairs = target_bins[:, 0, [4, 5]].cpu().numpy()
    draws = sampled_grids[:, 0, [4, 5]].cpu().numpy()
    supported = within_expert_support(draws, expert_pairs, 8.0)
    jitter = float(np.mean([
        temporal_jitter(grid) for grid in sampled_grids.cpu().numpy()
    ]))

    fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
    axes[0].plot([row['step'] for row in history],
                 [row['loss'] for row in history], label='CE')
    axes[0].plot([row['step'] for row in history],
                 [row['entropy'] for row in history], label='entropy')
    axes[0].axhline(math.log(ACTION_BINS), ls='--', c='grey', lw=1,
                    label='ln(B)')
    axes[0].set(xlabel='step', title=f'{name}: training')
    axes[0].legend()
    plot_action_distribution(
        logits[0, 0, 4].softmax(-1).cpu().numpy(), ax=axes[1])
    axes[1].set_title('marginal: timestep 0, control 4')
    plot_joint_support(expert_pairs, draws, supported, 8.0, ax=axes[2])
    axes[2].set_title(f'{name}: complete-grid draws')
    fig.tight_layout()
    plt.show()

    plot_chunk_comparison(prediction[0].numpy(), expert[0].numpy())
    plt.suptitle(f'{name}: held-out action chunk', y=1.01)
    plt.show()

    mae_std = metrics['mae_in_standard_deviations'].nanmean().item()
    print(f'{name}: validation CE={validation_ce:.3f}, '
          f'MAE/std={mae_std:.3f}, sampled jitter={jitter:.3f}')
    return {
        'head': head, 'backbone': backbone, 'history': history,
        'batch': batch, 'model_inputs': model_inputs,
        'target_bins': target_bins, 'validation_ce': validation_ce,
        'mae_std': mae_std, 'jitter': jitter, 'prediction': prediction,
        'logits': logits.cpu(), 'samples': sampled_grids.cpu(),
    }

### 4.5.1 Factorized head

The one-shot baseline of listing 4.2: one learned slot per grid cell, and
no cell's output enters any other cell's computation.

In [ ]:
TRAIN_STEPS = 10  # raise to 20_000 for a full, comparable run
results = {}
results['factorized'] = run_head_experiment('factorized',
                                            steps=TRAIN_STEPS)

### 4.5.2 Autoregressive head

Listings 4.3 and 4.4: teacher-forced training, then KV-cached generation
that decodes H x D positions in series. Fewer samples here, because that
serial loop is the whole point of the latency argument.

In [ ]:
results['autoregressive'] = run_head_experiment(
    'autoregressive', steps=TRAIN_STEPS, samples=32)

### 4.5.3 Parallel head (the shipped path)

Listing 4.5: H learned action positions appended to the Chapter 3 prefix,
bidirectional inside the action block, the whole grid in one forward pass.
This is the head sections 4.6 and 4.7 evaluate, and the one Chapter 5
extends with flow matching.

In [ ]:
results['parallel'] = run_head_experiment('parallel', steps=TRAIN_STEPS)

# The section 4.6 and 4.7 figures use the shipped parallel policy.
head = results['parallel']['head']
backbone = results['parallel']['backbone']
batch = results['parallel']['batch']
model_inputs = results['parallel']['model_inputs']

## 4.6 Visualizing the learnt action distribution

A falling loss says the policy puts more mass on expert bins. It does not
say what each marginal learned, or whether the cells go together.

In [ ]:
from ch04.analysis import (collect_cell_softmaxes, expert_pairs_from_batch,
                           joint_mismatch_samples, mismatch_rates,
                           neighborhood_softmax_figure,
                           sampled_grids_by_head, set_seed)
from ch04.diagnostics import (plot_joint_mismatch_panels,
                              plot_temporal_traces)

SEED, ANCHOR_INDEX, N_NEIGHBORS = 0, 0, 32
BASE_JOINT, PAIR_DIMS, TIMESTEP = 0, (4, 5), 0
set_seed(SEED)

# Figure 4.8: the held-out softmax cluster around one anchor frame. Report
# the anchor, neighbour count, and seed with the plot -- the caption below
# is generated from them so it cannot drift from the run.
collected = collect_cell_softmaxes(
    head, backbone, validation_loader, stats, tokenizer, device,
    timestep=TIMESTEP, control=BASE_JOINT, max_batches=8)
neighborhood_softmax_figure(
    collected, anchor_index=ANCHOR_INDEX,
    n_neighbors=min(N_NEIGHBORS, collected['states'].shape[0]),
    checkpoint=f'{CHECKPOINT_ROOT}/parallel/best.pt', seed=SEED)
plt.show()

### 4.6.2 Measuring joint mismatch

All three heads are trained now, so figure 4.9 compares real policies.
Each head samples through its own inference path. The autoregressive head
decodes 96 positions in series, so `JOINT_SAMPLES` dominates the runtime.

In [ ]:
JOINT_SAMPLES = 128  # the AR head decodes H x D positions per draw
set_seed(SEED)
trained_heads = {name: results[name]['head'] for name in HEAD_NAMES}
head_backbones = {name: results[name]['backbone'] for name in HEAD_NAMES}

# Figure 4.9: one panel per head, each on its own held-out observation.
pairs, grids = {}, {}
for name in HEAD_NAMES:
    inputs = results[name]['model_inputs']
    peer = {name: trained_heads[name]}
    pairs.update(joint_mismatch_samples(
        peer, head_backbones[name], inputs,
        dims=PAIR_DIMS, timestep=TIMESTEP, n_samples=JOINT_SAMPLES))
    grids.update(sampled_grids_by_head(
        peer, head_backbones[name], inputs, n_samples=12))

expert_pairs = expert_pairs_from_batch(
    batch, stats, tokenizer, device, dims=PAIR_DIMS, timestep=TIMESTEP)
plot_joint_mismatch_panels(pairs, expert_pairs, bin_range=(0, ACTION_BINS))
plt.show()
print('off-diagonal rate:',
      {k: round(v, 3) for k, v in
       mismatch_rates(pairs, ACTION_BINS // 2, ACTION_BINS // 2).items()})

# The same comparison along the temporal axis.
plot_temporal_traces(grids, control=PAIR_DIMS[0])
plt.show()
for name, grid in grids.items():
    print(f'{name:>15} temporal jitter: {temporal_jitter(grid[0]):.2f}')

## 4.7 From action distribution to controls

Decoding picks one bin per cell, the tokenizer returns normalized
midpoints, and Chapter 2's denormalizer converts those to the dataset's
raw command units. Section 4.7.2 then chooses how to run overlapping
chunks against the clock.

In [ ]:
from ch04.analysis import decoded_chunk_stream, open_loop_episode_trace
from ch04.diagnostics import (plot_execution_schedules,
                              plot_open_loop_episode)
from ch04.execution import execution_schedules

# Figure 4.10: the three section 4.7.2 schedules over one chunk stream.
chunks = decoded_chunk_stream(
    head, backbone, validation_loader, tokenizer, stats, device,
    max_batches=8)
plot_execution_schedules(execution_schedules(chunks), control=0)
plt.show()

# Figure 4.11: a held-out episode, expert against decoded commands.
trace = open_loop_episode_trace(
    head, backbone, validation_loader, tokenizer, stats, device,
    max_batches=8)
plot_open_loop_episode(trace['predicted'], trace['expert'], trace['valid'])
plt.show()

## Comparing the three learned policies

At ten steps these bars compare initialization noise, not designs. Set
`TRAIN_STEPS = 20_000` and re-run before reading anything into them.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
for axis, metric, title in zip(
        axes, ['validation_ce', 'mae_std', 'jitter'],
        ['Held-out CE', 'MAE / training std', 'Sampled jitter']):
    axis.bar(list(HEAD_NAMES),
             [results[name][metric] for name in HEAD_NAMES])
    axis.set_title(title)
    axis.tick_params(axis='x', rotation=20)
fig.tight_layout()
plt.show()
for name in HEAD_NAMES:
    result = results[name]
    print(f"{name:16s} CE={result['validation_ce']:.3f}  "
          f"MAE/std={result['mae_std']:.3f}  "
          f"jitter={result['jitter']:.3f}")

## Next experiments

- Re-run with `TRAIN_STEPS = 20_000` so the comparison above is meaningful.
- From a terminal the same three runs are one command:
  `ch04-train --head all --steps 20000`, and the figures are
  `ch04-figures checkpoints/parallel/best.pt --head parallel`.
- Sweep `N_BINS` and watch quantization error trade against the number of
  examples per bin.
- Section 4.7.4's closed-loop rollout is still a manuscript TODO: open-loop
  agreement here is not a task-success claim.